# Self-Organising Maps — Turin Project

This notebook trains two Self-Organising Maps (SOMs) on the vectorised Turin datasets:

- **Text SOM** — trained on sentence embeddings of the combined text corpus (British Pathé + IMDb + FCTP + Dialogues)
- **Image SOM** — trained on CLIP embeddings of the combined image corpus (FCTP building photos + IMDb film stills)

**Each SOM includes:**
- Quantisation Error (QE) and Topographic Error (TE) monitoring across training epochs
- Saving the SOM state after each epoch and selecting the best model based on QE and TE
- PCA-RGB visualisation using two different fitting approaches
- Minimum 10 search examples showing correspondences between the domains

All outputs are saved to `OUTPUT_SOM/`.

## 1. Setup

In [7]:
!pip install scikit-learn numpy pandas matplotlib seaborn tqdm -q


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\elena\AppData\Local\Programs\Python\Python314\python.exe -m pip install --upgrade pip


In [5]:
import os, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
from tqdm import tqdm
from pathlib import Path
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import copy

# --- Paths ---
VEC_OUT   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS"
SOM_OUT   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM"
FCTP_IMG  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_images"
IMDB_IMG  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\collection"

OUT_TEXT  = os.path.join(SOM_OUT, "text_som")
OUT_IMAGE = os.path.join(SOM_OUT, "image_som")
OUT_SEARCH= os.path.join(SOM_OUT, "search_examples")

for d in [OUT_TEXT, OUT_IMAGE, OUT_SEARCH]:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.size': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'figure.dpi': 150,
})

C_PATHE = '#C0462B'
C_IMDB  = '#1D6A72'
C_FCTP  = '#2A9199'
C_DIAL  = '#8B2E16'
SOURCE_COLORS = {'British Pathé': C_PATHE, 'IMDb': C_IMDB,
                 'FCTP': C_FCTP, 'Dialogue': C_DIAL,
                 'FCTP Buildings': '#D4860A', 'IMDb Stills': '#0A3D4A'}

def save(fig, folder, name):
    path = os.path.join(folder, name)
    fig.savefig(path, dpi=200, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'Saved: {path}')

print('Setup complete.')

Setup complete.


---
## 2. Load vectorised data

The vectors produced in the vectorisation notebook are loaded here. Sentence embeddings are used for the text SOM and CLIP embeddings for the image SOM — both selected because they capture semantic meaning rather than just surface-level features.

In [9]:
# Text vectors and metadata
X_text    = np.load(os.path.join(VEC_OUT, 'text', 'sentence_embeddings.npy'))
df_text   = pd.read_csv(os.path.join(VEC_OUT, 'text', 'text_metadata.csv'))
t_labels  = df_text['label'].tolist()
t_sources = df_text['source'].tolist()

# Image vectors and metadata
X_image   = np.load(os.path.join(VEC_OUT, 'image', 'clip_vectors.npy'))
df_image  = pd.read_csv(os.path.join(VEC_OUT, 'image', 'image_metadata.csv'))
i_labels  = df_image['label'].tolist()
i_sources = df_image['source'].tolist()
i_paths   = df_image['path'].tolist()

# Reduce dimensions with PCA before SOM for speed and quality
def reduce_pca(X, n=50):
    n = min(n, X.shape[1], X.shape[0] - 1)
    pca = PCA(n_components=n, random_state=42)
    return pca.fit_transform(X), pca

X_text_r,  pca_text  = reduce_pca(X_text,  50)
X_image_r, pca_image = reduce_pca(X_image, 50)

print(f'Text vectors  : {X_text.shape} → reduced to {X_text_r.shape}')
print(f'Image vectors : {X_image.shape} → reduced to {X_image_r.shape}')

Text vectors  : (573, 384) → reduced to (573, 50)
Image vectors : (2757, 512) → reduced to (2757, 50)


In [10]:
# Load full text content for display in the grid
import glob, os
from pathlib import Path

# Reload all original texts in the same order as the metadata
full_texts = []
df_meta = pd.read_csv(os.path.join(VEC_OUT, 'text', 'text_metadata.csv'))

# We need to reload texts from original sources
# Use the label to identify which text it is
# For now load from the combined corpus if saved
corpus_path = os.path.join(
    r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\MACHINE LEARNING\SAM_TEST_OUTPUTS\OUTPUTS",
    "combined_corpus.csv"
)

if os.path.exists(corpus_path):
    df_corpus = pd.read_csv(corpus_path)
    print(f'Corpus loaded: {len(df_corpus)} texts')
    # Match by position — corpus was built in same order
    full_texts = df_corpus['text'].tolist()
else:
    # Fallback to labels
    full_texts = t_labels
    print('Corpus not found — using labels')

print(f'Full texts available: {len(full_texts)}')

Corpus loaded: 541 texts
Full texts available: 541


---
## 3. SOM helper functions

### Quantisation Error and Topographic Error

**Quantisation Error (QE)** measures the average distance between each data point and its Best Matching Unit (BMU) on the SOM grid. A lower QE means the SOM cells are positioned closer to the actual data — the map is a more accurate representation of the data distribution. QE decreases as training progresses and the SOM adapts to the data.

**Topographic Error (TE)** measures how often the two best matching units for a data point are not adjacent on the SOM grid. A lower TE means the SOM preserves the topological structure of the data — similar items are not only mapped to similar cells but to cells that are physically close on the grid. TE is more sensitive to the SOM's organisation than QE.

In [11]:
class NumpySOM:
    def __init__(self, rows, cols, dim, sigma=1.0, lr=0.5, seed=42):
        np.random.seed(seed)
        self.rows    = rows
        self.cols    = cols
        self.dim     = dim
        self.sigma   = sigma
        self.lr      = lr
        self._weights = np.random.randn(rows, cols, dim).astype(np.float32)
        r, c = np.mgrid[0:rows, 0:cols]
        self._grid = np.stack([r.ravel(), c.ravel()], axis=1).astype(np.float32)

    def get_weights(self):
        return self._weights

    def winner(self, x):
        diff = self._weights - x
        dist = np.sum(diff ** 2, axis=2)
        return np.unravel_index(np.argmin(dist), dist.shape)

    def _neighbourhood(self, bmu_row, bmu_col, sigma):
        bmu = np.array([bmu_row, bmu_col], dtype=np.float32)
        d2  = np.sum((self._grid - bmu) ** 2, axis=1)
        h   = np.exp(-d2 / (2 * sigma ** 2))
        return h.reshape(self.rows, self.cols)

    def train_single(self, x, t, T):
        lr_t    = self.lr    * np.exp(-t / T)
        sigma_t = self.sigma * np.exp(-t / T)
        bmu     = self.winner(x)
        h       = self._neighbourhood(bmu[0], bmu[1], sigma_t)
        self._weights += lr_t * h[:, :, np.newaxis] * (x - self._weights)

    def train(self, data, n_iter):
        T = n_iter
        for t in range(T):
            idx = np.random.randint(0, len(data))
            self.train_single(data[idx], t, T)

    def random_weights_init(self, data):
        idx = np.random.choice(len(data), self.rows * self.cols, replace=True)
        self._weights = data[idx].reshape(self.rows, self.cols, self.dim).copy()


def quantisation_error(som, data):
    errors = []
    for x in data:
        bmu = som.winner(x)
        w   = som.get_weights()[bmu[0], bmu[1]]
        errors.append(np.linalg.norm(x - w))
    return float(np.mean(errors))


def topographic_error(som, data):
    errors = 0
    W = som.get_weights()
    for x in data:
        dists = np.sum((W - x) ** 2, axis=2)
        flat  = dists.ravel()
        i1    = np.argmin(flat)
        flat2 = flat.copy()
        flat2[i1] = np.inf
        i2    = np.argmin(flat2)
        r1, c1 = np.unravel_index(i1, dists.shape)
        r2, c2 = np.unravel_index(i2, dists.shape)
        if abs(r1 - r2) > 1 or abs(c1 - c2) > 1:
            errors += 1
    return errors / len(data)


def train_som_with_monitoring(data, som_size, sigma, lr, n_epochs, name):
    input_dim  = data.shape[1]
    n_samples  = len(data)
    iterations = n_samples

    som = NumpySOM(som_size, som_size, input_dim,
                   sigma=sigma, lr=lr, seed=42)
    som.random_weights_init(data)

    qe_history    = []
    te_history    = []
    saved_weights = []
    best_epoch    = 0
    best_score    = float('inf')

    te_sample = data[np.random.choice(len(data), min(100, len(data)), replace=False)]

    print(f'Training {name} SOM ({som_size}x{som_size}, {n_epochs} epochs)...')
    for epoch in tqdm(range(n_epochs), desc=f'{name} SOM'):
        idx = np.random.permutation(len(data))
        som.train(data[idx], iterations)

        qe = quantisation_error(som, data)
        te = topographic_error(som, te_sample)
        qe_history.append(qe)
        te_history.append(te)
        saved_weights.append(som.get_weights().copy())

        score = qe + te
        if score < best_score:
            best_score = score
            best_epoch = epoch

    print(f'Best epoch: {best_epoch+1} '
          f'(QE={qe_history[best_epoch]:.4f}, TE={te_history[best_epoch]:.4f})')

    som._weights = saved_weights[best_epoch]
    return som, qe_history, te_history, best_epoch


print('NumpySOM defined.')

NumpySOM defined.


---
## 4. Train the Text SOM

In [43]:
SOM_SIZE_TEXT = 20   # 20x20 grid = 400 cells for ~600 documents
EPOCHS_TEXT   = 20

som_text, qe_text, te_text, best_text = train_som_with_monitoring(
    data      = X_text_r.copy(),
    som_size  = SOM_SIZE_TEXT,
    sigma     = 3.0,
    lr        = 0.5,
    n_epochs  = EPOCHS_TEXT,
    name      = 'Text'
)

np.save(os.path.join(OUT_TEXT, 'text_som_weights.npy'), som_text.get_weights())
print('Text SOM weights saved.')

Training Text SOM (20x20, 20 epochs)...


Text SOM: 100%|████████████████████████████████████████████████████████████████████████| 20/20 [00:01<00:00, 11.47it/s]

Best epoch: 11 (QE=0.4917, TE=0.0100)
Text SOM weights saved.


### 4.1 QE and TE training plots

The plots below show how Quantisation Error and Topographic Error evolve during training. The vertical line marks the epoch selected as the best model — the point where the combined score (QE + TE) is lowest, representing the best balance between accuracy and topological organisation.

In [44]:
epochs = list(range(1, EPOCHS_TEXT + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs, qe_text, color=C_FCTP, linewidth=2, marker='o', markersize=4)
ax1.axvline(best_text + 1, color=C_PATHE, linestyle='--',
            linewidth=1.2, label=f'Best epoch ({best_text+1})')
ax1.set_title('Text SOM — Quantisation Error', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('QE')
ax1.legend()

ax2.plot(epochs, te_text, color=C_IMDB, linewidth=2, marker='o', markersize=4)
ax2.axvline(best_text + 1, color=C_PATHE, linestyle='--',
            linewidth=1.2, label=f'Best epoch ({best_text+1})')
ax2.set_title('Text SOM — Topographic Error', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('TE')
ax2.legend()

fig.suptitle('Text SOM Training History\n'
             'QE measures accuracy; TE measures topological organisation',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
save(fig, OUT_TEXT, 'text_som_qe_te.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\text_som\text_som_qe_te.png


### 4.2 Text SOM — source distribution map

Each cell of the SOM grid is coloured by the dominant source dataset of the documents mapped there. This shows how the four text sources distribute across the learned semantic space.

In [45]:
# Map each document to its BMU
bmu_text = [som_text.winner(x) for x in X_text_r]

# Build a grid showing dominant source per cell
grid_source = np.full((SOM_SIZE_TEXT, SOM_SIZE_TEXT), '', dtype=object)
grid_counts = {src: np.zeros((SOM_SIZE_TEXT, SOM_SIZE_TEXT))
               for src in set(t_sources)}

for bmu, src in zip(bmu_text, t_sources):
    grid_counts[src][bmu[0], bmu[1]] += 1

# Colour each cell by dominant source
rgb_grid = np.ones((SOM_SIZE_TEXT, SOM_SIZE_TEXT, 3))
src_list = list(SOURCE_COLORS.keys())

def hex_to_rgb(h):
    h = h.lstrip('#')
    return tuple(int(h[i:i+2], 16)/255 for i in (0, 2, 4))

for i in range(SOM_SIZE_TEXT):
    for j in range(SOM_SIZE_TEXT):
        counts = {src: grid_counts[src][i, j] for src in grid_counts}
        total  = sum(counts.values())
        if total > 0:
            dom = max(counts, key=counts.get)
            if dom in SOURCE_COLORS:
                rgb_grid[i, j] = hex_to_rgb(SOURCE_COLORS[dom])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(rgb_grid, origin='upper')

# Overlay document counts as text
for i in range(SOM_SIZE_TEXT):
    for j in range(SOM_SIZE_TEXT):
        total = int(sum(grid_counts[s][i,j] for s in grid_counts))
        if total > 0:
            ax.text(j, i, str(total), ha='center', va='center',
                    fontsize=5, color='white', alpha=0.8)

handles = [mpatches.Patch(color=v, label=k)
           for k, v in SOURCE_COLORS.items() if k in set(t_sources)]
ax.legend(handles=handles, loc='lower right',
          framealpha=0.9, fontsize=9)
ax.set_title(f'Text SOM — Source Distribution Map\n'
             f'Best epoch: {best_text+1} | Grid: {SOM_SIZE_TEXT}×{SOM_SIZE_TEXT}',
             fontweight='bold', pad=12)
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
save(fig, OUT_TEXT, 'text_som_source_map.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\text_som\text_som_source_map.png


### 4.3 Text SOM — PCA-RGB visualisation (two methods)

PCA-RGB visualises the SOM by mapping the three principal components of the weight vectors to the RGB colour channels. This reveals the internal structure of the trained map — regions of similar colour share similar learned representations.

**Method 1: PCA fitted on SOM cells** — the PCA is computed from the SOM weight vectors themselves. This shows the internal structure of the map as the SOM learned it.

**Method 2: PCA fitted on training data, applied to SOM cells** — the PCA is computed from all training documents, then applied to the SOM weights. This projects the SOM into the same space as the data, making the colours more directly interpretable in terms of the original data distribution.

The two methods often produce different colour gradients: Method 1 reflects the SOM's own organisation; Method 2 reflects how the SOM cells relate to the actual data spread.

In [46]:
weights_text = som_text.get_weights()
W_flat       = weights_text.reshape(-1, weights_text.shape[2])

# Method 1 — PCA fitted on SOM cells
pca1 = PCA(n_components=3, random_state=42)
rgb1 = pca1.fit_transform(W_flat)
rgb1 = (rgb1 - rgb1.min()) / (rgb1.max() - rgb1.min() + 1e-9)
rgb1_grid = rgb1.reshape(SOM_SIZE_TEXT, SOM_SIZE_TEXT, 3)

# Method 2 — PCA fitted on training data, applied to SOM cells
pca2 = PCA(n_components=3, random_state=42)
pca2.fit(X_text_r)
rgb2 = pca2.transform(W_flat)
rgb2 = (rgb2 - rgb2.min()) / (rgb2.max() - rgb2.min() + 1e-9)
rgb2_grid = rgb2.reshape(SOM_SIZE_TEXT, SOM_SIZE_TEXT, 3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))

ax1.imshow(rgb1_grid, origin='upper')
ax1.set_title('PCA-RGB — Method 1\nPCA fitted on SOM cell weights',
              fontweight='bold', pad=10)
ax1.set_xticks([]); ax1.set_yticks([])

ax2.imshow(rgb2_grid, origin='upper')
ax2.set_title('PCA-RGB — Method 2\nPCA fitted on training data, applied to SOM cells',
              fontweight='bold', pad=10)
ax2.set_xticks([]); ax2.set_yticks([])

fig.suptitle('Text SOM — PCA-RGB Visualisation (two methods)',
             fontsize=12, fontweight='bold', y=1.01)
fig.tight_layout()
save(fig, OUT_TEXT, 'text_som_pca_rgb.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\text_som\text_som_pca_rgb.png


In [47]:
# Text SOM — clean readable text, no colours
DISPLAY_SIZE = 10

fig, axes = plt.subplots(DISPLAY_SIZE, DISPLAY_SIZE, figsize=(26, 26))
fig.patch.set_facecolor('white')

bmu_text = [som_text.winner(x) for x in X_text_r]

cell_contents = {}
for idx, bmu in enumerate(bmu_text):
    key = (bmu[0], bmu[1])
    if key not in cell_contents:
        cell_contents[key] = []
    cell_contents[key].append(idx)

# Find densest 10x10 region
best_r, best_c, best_count = 0, 0, 0
for r in range(SOM_SIZE_TEXT - DISPLAY_SIZE):
    for c in range(SOM_SIZE_TEXT - DISPLAY_SIZE):
        count = sum(1 for i in range(r, r+DISPLAY_SIZE)
                    for j in range(c, c+DISPLAY_SIZE)
                    if (i,j) in cell_contents)
        if count > best_count:
            best_count = count
            best_r, best_c = r, c

for pi, i in enumerate(range(best_r, best_r + DISPLAY_SIZE)):
    for pj, j in enumerate(range(best_c, best_c + DISPLAY_SIZE)):
        ax = axes[pi][pj]
        ax.set_xticks([]); ax.set_yticks([])
        ax.set_facecolor('white')
        for spine in ax.spines.values():
            spine.set_linewidth(0.4)
            spine.set_edgecolor('#ccc')

        key = (i, j)
        if key in cell_contents:
            idx          = cell_contents[key][0]
            src          = t_sources[idx]
            text_content = full_texts[idx] if idx < len(full_texts) else t_labels[idx]
            label        = text_content[:200]

            words = label.split()
            lines, line = [], ''
            for w in words:
                if len(line) + len(w) > 20:
                    lines.append(line.strip())
                    line = w + ' '
                else:
                    line += w + ' '
            if line:
                lines.append(line.strip())
            lines = lines[:7]

            ax.text(0.5, 0.97, src[:12],
                    ha='center', va='top',
                    fontsize=5.5, color='#aaa',
                    transform=ax.transAxes)

            ax.text(0.5, 0.48, '\n'.join(lines),
                    ha='center', va='center',
                    fontsize=6.0, color='#111',
                    transform=ax.transAxes,
                    linespacing=1.3)

            if len(cell_contents[key]) > 1:
                ax.text(0.97, 0.03,
                        f'+{len(cell_contents[key])-1}',
                        ha='right', va='bottom',
                        fontsize=5, color='#bbb',
                        transform=ax.transAxes)
        else:
            ax.set_facecolor('#f5f5f5')

fig.suptitle('Text SOM — Semantic Grid\n'
             'Each cell = one document placed by semantic similarity',
             fontsize=13, fontweight='bold', y=1.01)
fig.tight_layout(pad=0.4)
save(fig, OUT_TEXT, 'text_som_content_grid.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\text_som\text_som_content_grid.png


---
## 5. Train the Image SOM

In [10]:
SOM_SIZE_IMG = 18   # 18x18 grid for image corpus
EPOCHS_IMG   = 20

som_image, qe_image, te_image, best_image = train_som_with_monitoring(
    data      = X_image_r.copy(),
    som_size  = SOM_SIZE_IMG,
    sigma     = 3.0,
    lr        = 0.5,
    n_epochs  = EPOCHS_IMG,
    name      = 'Image'
)

np.save(os.path.join(OUT_IMAGE, 'image_som_weights.npy'), som_image.get_weights())
print('Image SOM weights saved.')

Training Image SOM (18x18, 20 epochs)...


Image SOM: 100%|███████████████████████████████████████████████████████████████████████| 20/20 [00:06<00:00,  2.89it/s]

Best epoch: 17 (QE=0.3366, TE=0.0200)
Image SOM weights saved.


### 5.1 Image SOM — QE and TE training plots

In [11]:
epochs_img = list(range(1, EPOCHS_IMG + 1))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

ax1.plot(epochs_img, qe_image, color=C_FCTP, linewidth=2, marker='o', markersize=4)
ax1.axvline(best_image + 1, color=C_PATHE, linestyle='--',
            linewidth=1.2, label=f'Best epoch ({best_image+1})')
ax1.set_title('Image SOM — Quantisation Error', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('QE')
ax1.legend()

ax2.plot(epochs_img, te_image, color=C_IMDB, linewidth=2, marker='o', markersize=4)
ax2.axvline(best_image + 1, color=C_PATHE, linestyle='--',
            linewidth=1.2, label=f'Best epoch ({best_image+1})')
ax2.set_title('Image SOM — Topographic Error', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('TE')
ax2.legend()

fig.suptitle('Image SOM Training History',
             fontsize=11, fontweight='bold', y=1.02)
fig.tight_layout()
save(fig, OUT_IMAGE, 'image_som_qe_te.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\image_som\image_som_qe_te.png


### 5.2 Image SOM — source distribution map

In [12]:
bmu_image = [som_image.winner(x) for x in X_image_r]

IMG_SOURCES = {'FCTP Buildings': C_FCTP, 'IMDb Stills': C_IMDB}
grid_img = {src: np.zeros((SOM_SIZE_IMG, SOM_SIZE_IMG))
            for src in IMG_SOURCES}

for bmu, src in zip(bmu_image, i_sources):
    if src in grid_img:
        grid_img[src][bmu[0], bmu[1]] += 1

rgb_img = np.ones((SOM_SIZE_IMG, SOM_SIZE_IMG, 3))
for i in range(SOM_SIZE_IMG):
    for j in range(SOM_SIZE_IMG):
        counts = {src: grid_img[src][i, j] for src in grid_img}
        total  = sum(counts.values())
        if total > 0:
            dom = max(counts, key=counts.get)
            rgb_img[i, j] = hex_to_rgb(IMG_SOURCES[dom])

fig, ax = plt.subplots(figsize=(9, 9))
ax.imshow(rgb_img, origin='upper')
for i in range(SOM_SIZE_IMG):
    for j in range(SOM_SIZE_IMG):
        total = int(sum(grid_img[s][i,j] for s in grid_img))
        if total > 0:
            ax.text(j, i, str(total), ha='center', va='center',
                    fontsize=5.5, color='white', alpha=0.85)

handles = [mpatches.Patch(color=v, label=k) for k, v in IMG_SOURCES.items()]
ax.legend(handles=handles, loc='lower right', framealpha=0.9, fontsize=9)
ax.set_title(f'Image SOM — Source Distribution Map\n'
             f'Best epoch: {best_image+1} | Grid: {SOM_SIZE_IMG}×{SOM_SIZE_IMG}',
             fontweight='bold', pad=12)
ax.set_xticks([]); ax.set_yticks([])
fig.tight_layout()
save(fig, OUT_IMAGE, 'image_som_source_map.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\image_som\image_som_source_map.png


### 5.3 Image SOM — PCA-RGB visualisation (two methods)

In [13]:
weights_img = som_image.get_weights()
W_img_flat  = weights_img.reshape(-1, weights_img.shape[2])

# Method 1
pca_img1 = PCA(n_components=3, random_state=42)
rgb_img1 = pca_img1.fit_transform(W_img_flat)
rgb_img1 = (rgb_img1 - rgb_img1.min()) / (rgb_img1.max() - rgb_img1.min() + 1e-9)
rgb_img1_grid = rgb_img1.reshape(SOM_SIZE_IMG, SOM_SIZE_IMG, 3)

# Method 2
pca_img2 = PCA(n_components=3, random_state=42)
pca_img2.fit(X_image_r)
rgb_img2 = pca_img2.transform(W_img_flat)
rgb_img2 = (rgb_img2 - rgb_img2.min()) / (rgb_img2.max() - rgb_img2.min() + 1e-9)
rgb_img2_grid = rgb_img2.reshape(SOM_SIZE_IMG, SOM_SIZE_IMG, 3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 7))
ax1.imshow(rgb_img1_grid, origin='upper')
ax1.set_title('PCA-RGB — Method 1\nPCA fitted on SOM cell weights',
              fontweight='bold', pad=10)
ax1.set_xticks([]); ax1.set_yticks([])

ax2.imshow(rgb_img2_grid, origin='upper')
ax2.set_title('PCA-RGB — Method 2\nPCA fitted on training data, applied to SOM cells',
              fontweight='bold', pad=10)
ax2.set_xticks([]); ax2.set_yticks([])

fig.suptitle('Image SOM — PCA-RGB Visualisation (two methods)',
             fontsize=12, fontweight='bold', y=1.01)
fig.tight_layout()
save(fig, OUT_IMAGE, 'image_som_pca_rgb.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\image_som\image_som_pca_rgb.png


In [20]:
import cv2
from pathlib import Path

# Image SOM — visual grid showing thumbnails per cell
THUMB = 120  # thumbnail size in pixels

bmu_image = [som_image.winner(x) for x in X_image_r]

cell_img = {}
for idx, bmu in enumerate(bmu_image):
    key = (bmu[0], bmu[1])
    if key not in cell_img:
        cell_img[key] = idx  # keep first image per cell

# Build canvas
canvas = np.ones((SOM_SIZE_IMG * THUMB, SOM_SIZE_IMG * THUMB, 3),
                  dtype=np.uint8) * 240

for i in range(SOM_SIZE_IMG):
    for j in range(SOM_SIZE_IMG):
        key = (i, j)
        y0, x0 = i * THUMB, j * THUMB
        if key in cell_img:
            path = i_paths[cell_img[key]]
            img  = cv2.imread(path)
            if img is not None:
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                img = cv2.resize(img, (THUMB, THUMB))
                canvas[y0:y0+THUMB, x0:x0+THUMB] = img
            else:
                canvas[y0:y0+THUMB, x0:x0+THUMB] = 180
        # Draw grid lines
        canvas[y0, x0:x0+THUMB] = [255, 255, 255]
        canvas[y0:y0+THUMB, x0] = [255, 255, 255]

fig, ax = plt.subplots(figsize=(13, 13))
ax.imshow(canvas)
ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values():
    spine.set_visible(False)

handles = [mpatches.Patch(color=SOURCE_COLORS.get(k, '#aaa'), label=k)
           for k in ['FCTP Buildings', 'IMDb Stills']]
ax.legend(handles=handles, loc='lower right',
          framealpha=0.9, fontsize=9)

ax.set_title(f'Image SOM — Visual Grid (18×18)\n'
             f'Each cell = one image | border colour = source dataset',
             fontsize=12, fontweight='bold', pad=12)
fig.tight_layout()
save(fig, OUT_IMAGE, 'image_som_visual_grid.png')

Saved: C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM\image_som\image_som_visual_grid.png


---
## 6. Search examples — Text SOM

Each search activates the Text SOM with a query document and retrieves the most similar documents from all four sources. This demonstrates correspondences between the datasets — showing which Pathé clips, films, buildings, and dialogue locations share the same semantic region of the map.

In [14]:
# Reload everything needed for search examples
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import os

# --- Paths ---
VEC_OUT   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\VECTORIZATION\OUTPUTS"
SOM_OUT   = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\SOM\OUTPUT_SOM"
FCTP_IMG  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\INDUSTRIAL BUILDINGS\fctp_images"
IMDB_IMG  = r"C:\_FILES\Desktop\RC11\NEW\TERM 1\DIGITAL SKILLS\WEB_SCRAPING\IMDB\collection"

OUT_TEXT  = os.path.join(SOM_OUT, "text_som")
OUT_IMAGE = os.path.join(SOM_OUT, "image_som")
OUT_SEARCH= os.path.join(SOM_OUT, "search_examples")

# --- Colours ---
SOURCE_COLORS = {
    'British Pathé':  '#C0462B',
    'IMDb':           '#0A3D4A',
    'FCTP':           '#D4860A',
    'Dialogue':       '#6B1A1A',
    'FCTP Buildings': '#D4860A',
    'IMDb Stills':    '#0A3D4A',
}

# --- SOM sizes ---
SOM_SIZE_TEXT = 20
SOM_SIZE_IMG  = 18

# --- Load text data ---
X_text_raw = np.load(os.path.join(VEC_OUT, 'text', 'sentence_embeddings.npy'))
df_text    = pd.read_csv(os.path.join(VEC_OUT, 'text', 'text_metadata.csv'))
t_labels   = df_text['label'].tolist()
t_sources  = df_text['source'].tolist()

# --- Load image data ---
X_image_raw = np.load(os.path.join(VEC_OUT, 'image', 'clip_vectors.npy'))
df_image    = pd.read_csv(os.path.join(VEC_OUT, 'image', 'image_metadata.csv'))
i_labels    = df_image['label'].tolist()
i_sources   = df_image['source'].tolist()
i_paths     = df_image['path'].tolist()

# --- PCA reduction ---
def reduce_pca(X, n=50):
    n = min(n, X.shape[1], X.shape[0]-1)
    return PCA(n_components=n, random_state=42).fit_transform(X)

X_text_r  = reduce_pca(X_text_raw, 50)
X_image_r = reduce_pca(X_image_raw, 50)

# --- Reload SOM weights ---
som_text  = NumpySOM(SOM_SIZE_TEXT, SOM_SIZE_TEXT, X_text_r.shape[1],
                     sigma=3.0, lr=0.5, seed=42)
som_text._weights = np.load(os.path.join(OUT_TEXT, 'text_som_weights.npy'))

som_image = NumpySOM(SOM_SIZE_IMG, SOM_SIZE_IMG, X_image_r.shape[1],
                     sigma=3.0, lr=0.5, seed=42)
som_image._weights = np.load(os.path.join(OUT_IMAGE, 'image_som_weights.npy'))

print(f'Text  vectors : {X_text_r.shape}')
print(f'Image vectors : {X_image_r.shape}')
print(f'Text  sources : {set(t_sources)}')
print(f'Image sources : {set(i_sources)}')
print('Ready.')

Text  vectors : (573, 50)
Image vectors : (2757, 50)
Text  sources : {'FCTP', 'British Pathé', 'Dialogue', 'IMDb'}
Image sources : {'IMDb Stills', 'FCTP Buildings'}
Ready.


In [20]:
# Fit PCA on original full-dimension vectors — do this once before searches
from sklearn.decomposition import PCA
from sentence_transformers import SentenceTransformer

model    = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')
pca_text = PCA(n_components=50, random_state=42)
pca_text.fit(np.load(os.path.join(VEC_OUT, 'text', 'sentence_embeddings.npy')))

pca_image = PCA(n_components=50, random_state=42)
pca_image.fit(np.load(os.path.join(VEC_OUT, 'image', 'clip_vectors.npy')))

print('PCA models fitted.')

PCA models fitted.


In [36]:
from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import matplotlib.pyplot as plt
import os

X_text_full  = np.load(os.path.join(VEC_OUT, 'text', 'sentence_embeddings.npy'))
pca_text_fit = PCA(n_components=50, random_state=42)
pca_text_fit.fit(X_text_full)

model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

def search_by_word(keyword, som, vectors, sources, labels,
                   som_size, source_colors, out_path):

    raw_vec   = model.encode([keyword])[0]
    query_vec = pca_text_fit.transform([raw_vec])[0]

    bmu     = som.winner(query_vec)
    bmu_all = [som.winner(x) for x in vectors]
    sims    = np.array([
        float(cosine_similarity([query_vec], [v])[0][0])
        for v in vectors
    ])

    max_sim = float(np.max(sims))
    sigma   = 2.0 + (1.0 - max_sim) * 4.0

   # Activation based on similarity to each SOM cell's weights
    W = som.get_weights()
    activation = np.zeros((som_size, som_size))
    for i in range(som_size):
        for j in range(som_size):
            cell_sim = float(cosine_similarity(
                [query_vec], [W[i, j]])[0][0])
            activation[i, j] = max(0, cell_sim)

    # Normalise
    activation = (activation - activation.min()) / (
        activation.max() - activation.min() + 1e-9)

    # Boost cells with high cosine similarity docs
    for idx, b in enumerate(bmu_all):
        if sims[idx] > activation[b[0], b[1]]:
            activation[b[0], b[1]] = max(
                activation[b[0], b[1]], sims[idx])

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.patch.set_facecolor('white')

    ax = axes[0]
    ax.set_facecolor('white')
    im = ax.imshow(activation, cmap='Blues', origin='upper',
                   vmin=0.2, vmax=1.0, interpolation='nearest')
    rect = plt.Rectangle(
        (bmu[1]-0.5, bmu[0]-0.5), 1, 1,
        linewidth=3, edgecolor='black', facecolor='none', zorder=5)
    ax.add_patch(rect)

    for i in range(som_size):
        for j in range(som_size):
            if activation[i, j] > 0.5:
                cell_docs = [k for k, b in enumerate(bmu_all)
                             if b[0]==i and b[1]==j]
                if cell_docs:
                    best = cell_docs[np.argmax([sims[k] for k in cell_docs])]
                    ax.text(j, i, labels[best][:12],
                            ha='center', va='center',
                            fontsize=3.5,
                            color='black',
                            fontweight='bold')

    plt.colorbar(im, ax=ax, shrink=0.6, label='Activation')
    ax.set_title(f'SOM Activation Map\nQuery word: "{keyword}"',
                 color='black', fontsize=9, pad=8)
    ax.set_xticks([]); ax.set_yticks([])

    ax2 = axes[1]
    ax2.set_facecolor('white')
    ax2.axis('off')
    top_idx = np.argsort(-sims)[:5]
    ax2.text(0.5, 0.97, 'Top Results', color='black',
             fontsize=11, fontweight='bold',
             ha='center', va='top', transform=ax2.transAxes)
    ax2.text(0.5, 0.90, f'Query word: "{keyword}"',
             color='black', fontsize=10, ha='center', va='top',
             transform=ax2.transAxes)
    for rank, idx in enumerate(top_idx):
        y   = 0.78 - rank * 0.14
        col = source_colors.get(sources[idx], '#333')
        ax2.text(0.05, y, f'{rank+1}. [{sources[idx]}]',
                 color=col, fontsize=8, fontweight='bold',
                 transform=ax2.transAxes)
        ax2.text(0.05, y-0.05, labels[idx][:55],
                 color='black', fontsize=7.5, transform=ax2.transAxes)
        ax2.text(0.92, y-0.02, f'sim={sims[idx]:.2f}',
                 color='#666', fontsize=7, ha='right',
                 transform=ax2.transAxes)

    fig.suptitle(f'Text SOM Search — "{keyword}"',
                 color='black', fontsize=11, fontweight='bold', y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  "{keyword}" → BMU: {bmu} | sigma: {sigma:.2f}')


# Run 10 text searches
text_keywords = ['fiat', 'strike', 'industrial', 'crime',
                 'ruin', 'silence', 'stone', 'memory', 'factory', 'river']

print('Running Text SOM searches...')
for n, kw in enumerate(text_keywords, 1):
    out = os.path.join(OUT_SEARCH, f'text_search_{n:02d}.png')
    search_by_word(kw, som_text, X_text_r, t_sources, t_labels,
                   SOM_SIZE_TEXT, SOURCE_COLORS, out)
print('Done.')

Running Text SOM searches...
  "fiat" → BMU: (np.int64(0), np.int64(4)) | sigma: 3.53
  "strike" → BMU: (np.int64(17), np.int64(11)) | sigma: 3.51
  "industrial" → BMU: (np.int64(12), np.int64(14)) | sigma: 3.63
  "crime" → BMU: (np.int64(5), np.int64(2)) | sigma: 2.94
  "ruin" → BMU: (np.int64(3), np.int64(19)) | sigma: 3.81
  "silence" → BMU: (np.int64(3), np.int64(19)) | sigma: 3.28
  "stone" → BMU: (np.int64(2), np.int64(17)) | sigma: 3.61
  "memory" → BMU: (np.int64(19), np.int64(11)) | sigma: 4.24
  "factory" → BMU: (np.int64(13), np.int64(14)) | sigma: 4.04
  "river" → BMU: (np.int64(14), np.int64(11)) | sigma: 4.18
Done.


---
## 7. Search examples — Image SOM

Each search activates the Image SOM with a query image and retrieves the most visually and semantically similar images from both FCTP buildings and IMDb film stills. This reveals which buildings and film scenes share the same atmospheric region of the map.

In [41]:
import open_clip
import torch

# Load CLIP once
clip_model, _, preprocess = open_clip.create_model_and_transforms(
    'ViT-B-32', pretrained='openai')
clip_model.eval()
tokenizer = open_clip.get_tokenizer('ViT-B-32')

# Load full image vectors and fit PCA once
X_image_full  = np.load(os.path.join(VEC_OUT, 'image', 'clip_vectors.npy'))
pca_image_fit = PCA(n_components=50, random_state=42)
pca_image_fit.fit(X_image_full)

C:\Users\elena\AppData\Local\Programs\Python\Python314\Lib\site-packages\open_clip\factory.py:450: UserWarning: QuickGELU mismatch between final model config (quick_gelu=False) and pretrained tag 'openai' (quick_gelu=True).
  warnings.warn(


,n_components,50
,copy,True
,whiten,False
,svd_solver,'auto'
,tol,0.0
,iterated_power,'auto'
,n_oversamples,10
,power_iteration_normalizer,'auto'
,random_state,42


In [42]:
def search_image_by_word(keyword, som, vectors, sources, labels,
                         som_size, source_colors, out_path):

    with torch.no_grad():
        text_tokens = tokenizer([keyword])
        raw_vec     = clip_model.encode_text(text_tokens).numpy()[0]

    query_vec = pca_image_fit.transform([raw_vec])[0]

    bmu     = som.winner(query_vec)
    bmu_all = [som.winner(x) for x in vectors]
    sims    = np.array([
        float(cosine_similarity([query_vec], [v])[0][0])
        for v in vectors
    ])

    # Activation based on similarity to each SOM cell's weights
    W = som.get_weights()
    activation = np.zeros((som_size, som_size))
    for i in range(som_size):
        for j in range(som_size):
            cell_sim = float(cosine_similarity(
                [query_vec], [W[i, j]])[0][0])
            activation[i, j] = max(0, cell_sim)

    activation = (activation - activation.min()) / (
        activation.max() - activation.min() + 1e-9)

    for idx, b in enumerate(bmu_all):
        if sims[idx] > activation[b[0], b[1]]:
            activation[b[0], b[1]] = max(
                activation[b[0], b[1]], sims[idx])

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    fig.patch.set_facecolor('white')

    ax = axes[0]
    ax.set_facecolor('white')
    im = ax.imshow(activation, cmap='Reds', origin='upper',
                   vmin=0.2, vmax=1.0, interpolation='nearest')
    rect = plt.Rectangle(
        (bmu[1]-0.5, bmu[0]-0.5), 1, 1,
        linewidth=3, edgecolor='black', facecolor='none', zorder=5)
    ax.add_patch(rect)

    for i in range(som_size):
        for j in range(som_size):
            if activation[i, j] > 0.5:
                cell_docs = [k for k, b in enumerate(bmu_all)
                             if b[0]==i and b[1]==j]
                if cell_docs:
                    best = cell_docs[np.argmax([sims[k] for k in cell_docs])]
                    ax.text(j, i, labels[best][:12],
                            ha='center', va='center',
                            fontsize=3.5, color='black',
                            fontweight='bold')

    plt.colorbar(im, ax=ax, shrink=0.6, label='Activation')
    ax.set_title(f'Image SOM Activation Map\nQuery word: "{keyword}"',
                 color='black', fontsize=9, pad=8)
    ax.set_xticks([]); ax.set_yticks([])

    ax2 = axes[1]
    ax2.set_facecolor('white')
    ax2.axis('off')
    top_idx = np.argsort(-sims)[:5]
    ax2.text(0.5, 0.97, 'Top Results', color='black',
             fontsize=11, fontweight='bold',
             ha='center', va='top', transform=ax2.transAxes)
    ax2.text(0.5, 0.90, f'Query word: "{keyword}"',
             color='black', fontsize=10, ha='center', va='top',
             transform=ax2.transAxes)
    for rank, idx in enumerate(top_idx):
        y   = 0.78 - rank * 0.14
        col = source_colors.get(sources[idx], '#333')
        ax2.text(0.05, y, f'{rank+1}. [{sources[idx]}]',
                 color=col, fontsize=8, fontweight='bold',
                 transform=ax2.transAxes)
        ax2.text(0.05, y-0.05, labels[idx][:55],
                 color='black', fontsize=7.5, transform=ax2.transAxes)
        ax2.text(0.92, y-0.02, f'sim={sims[idx]:.2f}',
                 color='#666', fontsize=7, ha='right',
                 transform=ax2.transAxes)

    fig.suptitle(f'Image SOM Search — "{keyword}"',
                 color='black', fontsize=11, fontweight='bold', y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=180, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f'  "{keyword}" → BMU: {bmu}')


# Run 10 image searches
image_keywords = ['facade', 'interior', 'factory', 'staircase',
                  'window', 'street', 'corridor', 'roof', 'arch', 'courtyard']

print('Running Image SOM searches...')
for n, kw in enumerate(image_keywords, 1):
    out = os.path.join(OUT_SEARCH, f'image_search_{n:02d}.png')
    search_image_by_word(kw, som_image, X_image_r, i_sources, i_labels,
                         SOM_SIZE_IMG, SOURCE_COLORS, out)
print('Done.')

Running Image SOM searches...
  "facade" → BMU: (np.int64(17), np.int64(13))
  "interior" → BMU: (np.int64(4), np.int64(17))
  "factory" → BMU: (np.int64(8), np.int64(8))
  "staircase" → BMU: (np.int64(8), np.int64(17))
  "window" → BMU: (np.int64(8), np.int64(17))
  "street" → BMU: (np.int64(17), np.int64(17))
  "corridor" → BMU: (np.int64(4), np.int64(17))
  "roof" → BMU: (np.int64(5), np.int64(3))
  "arch" → BMU: (np.int64(17), np.int64(11))
  "courtyard" → BMU: (np.int64(17), np.int64(14))
Done.


---
## 8. Model selection justification

This cell prints the QE and TE at each epoch and explains why the selected epoch was chosen as the best model.

In [16]:
print('TEXT SOM — Epoch-by-epoch QE and TE')
print(f'{"Epoch":>6} {"QE":>10} {"TE":>10} {"Combined":>12}')
print('-' * 42)
for ep, (qe, te) in enumerate(zip(qe_text, te_text), 1):
    marker = ' ← BEST' if ep == best_text + 1 else ''
    print(f'{ep:>6} {qe:>10.4f} {te:>10.4f} {qe+te:>12.4f}{marker}')

print()
print('IMAGE SOM — Epoch-by-epoch QE and TE')
print(f'{"Epoch":>6} {"QE":>10} {"TE":>10} {"Combined":>12}')
print('-' * 42)
for ep, (qe, te) in enumerate(zip(qe_image, te_image), 1):
    marker = ' ← BEST' if ep == best_image + 1 else ''
    print(f'{ep:>6} {qe:>10.4f} {te:>10.4f} {qe+te:>12.4f}{marker}')

print()
print('Model selection rationale:')
print('The best epoch is selected as the one with the lowest combined score')
print('(QE + TE), representing the best balance between accuracy and topology.')
print('This avoids selecting a model that is accurate but topologically disordered,')
print('or one that preserves topology at the cost of poor representation quality.')

TEXT SOM — Epoch-by-epoch QE and TE
 Epoch         QE         TE     Combined
------------------------------------------
     1     0.4930     0.0700       0.5630
     2     0.4952     0.1000       0.5952
     3     0.4959     0.0300       0.5259
     4     0.4887     0.0300       0.5187
     5     0.4948     0.0300       0.5248
     6     0.4898     0.0500       0.5398
     7     0.4974     0.0600       0.5574
     8     0.4929     0.0500       0.5429
     9     0.4940     0.0400       0.5340
    10     0.4954     0.0100       0.5054
    11     0.4917     0.0100       0.5017 ← BEST
    12     0.4973     0.0500       0.5473
    13     0.4913     0.0600       0.5513
    14     0.4946     0.0600       0.5546
    15     0.4919     0.0600       0.5519
    16     0.4903     0.0200       0.5103
    17     0.4946     0.0200       0.5146
    18     0.4959     0.0500       0.5459
    19     0.4911     0.0200       0.5111
    20     0.4919     0.0600       0.5519

IMAGE SOM — Epoch-by-epoch QE a

---
## 9. Summary

In [17]:
import os
print('=' * 55)
print('SOM ANALYSIS SUMMARY')
print('=' * 55)
print(f'Text SOM  : {SOM_SIZE_TEXT}×{SOM_SIZE_TEXT} | Best epoch: {best_text+1}/{EPOCHS_TEXT}')
print(f'  QE: {qe_text[best_text]:.4f} | TE: {te_text[best_text]:.4f}')
print(f'Image SOM : {SOM_SIZE_IMG}×{SOM_SIZE_IMG} | Best epoch: {best_image+1}/{EPOCHS_IMG}')
print(f'  QE: {qe_image[best_image]:.4f} | TE: {te_image[best_image]:.4f}')
print()
all_outputs = []
for folder in [OUT_TEXT, OUT_IMAGE, OUT_SEARCH]:
    files = [f for f in os.listdir(folder) if f.endswith('.png') or f.endswith('.npy')]
    folder_name = os.path.basename(folder)
    print(f'{folder_name}/')
    for f in sorted(files):
        print(f'  - {f}')
print('=' * 55)

SOM ANALYSIS SUMMARY
Text SOM  : 20×20 | Best epoch: 11/20
  QE: 0.4917 | TE: 0.0100
Image SOM : 18×18 | Best epoch: 17/20
  QE: 0.3366 | TE: 0.0200

text_som/
  - text_som_pca_rgb.png
  - text_som_qe_te.png
  - text_som_source_map.png
  - text_som_weights.npy
image_som/
  - image_som_pca_rgb.png
  - image_som_qe_te.png
  - image_som_source_map.png
  - image_som_weights.npy
search_examples/
  - image_search_01.png
  - image_search_02.png
  - image_search_03.png
  - image_search_04.png
  - image_search_05.png
  - image_search_06.png
  - image_search_07.png
  - image_search_08.png
  - image_search_09.png
  - image_search_10.png
  - text_search_01.png
  - text_search_02.png
  - text_search_03.png
  - text_search_04.png
  - text_search_05.png
  - text_search_06.png
  - text_search_07.png
  - text_search_08.png
  - text_search_09.png
  - text_search_10.png


In [48]:
print('TEXT SOM')
print(f'Best epoch: {best_text+1}/{EPOCHS_TEXT}')
print(f'QE: {qe_text[best_text]:.4f}')
print(f'TE: {te_text[best_text]:.4f}')
print()
print('Epoch-by-epoch:')
for ep, (qe, te) in enumerate(zip(qe_text, te_text), 1):
    marker = ' ← BEST' if ep == best_text+1 else ''
    print(f'  Epoch {ep:2d}: QE={qe:.4f}  TE={te:.4f}{marker}')

print()
print('IMAGE SOM')
print(f'Best epoch: {best_image+1}/{EPOCHS_IMG}')
print(f'QE: {qe_image[best_image]:.4f}')
print(f'TE: {te_image[best_image]:.4f}')
print()
print('Epoch-by-epoch:')
for ep, (qe, te) in enumerate(zip(qe_image, te_image), 1):
    marker = ' ← BEST' if ep == best_image+1 else ''
    print(f'  Epoch {ep:2d}: QE={qe:.4f}  TE={te:.4f}{marker}')

TEXT SOM
Best epoch: 11/20
QE: 0.4917
TE: 0.0100

Epoch-by-epoch:
  Epoch  1: QE=0.4930  TE=0.0700
  Epoch  2: QE=0.4952  TE=0.1000
  Epoch  3: QE=0.4959  TE=0.0300
  Epoch  4: QE=0.4887  TE=0.0300
  Epoch  5: QE=0.4948  TE=0.0300
  Epoch  6: QE=0.4898  TE=0.0500
  Epoch  7: QE=0.4974  TE=0.0600
  Epoch  8: QE=0.4929  TE=0.0500
  Epoch  9: QE=0.4940  TE=0.0400
  Epoch 10: QE=0.4954  TE=0.0100
  Epoch 11: QE=0.4917  TE=0.0100 ← BEST
  Epoch 12: QE=0.4973  TE=0.0500
  Epoch 13: QE=0.4913  TE=0.0600
  Epoch 14: QE=0.4946  TE=0.0600
  Epoch 15: QE=0.4919  TE=0.0600
  Epoch 16: QE=0.4903  TE=0.0200
  Epoch 17: QE=0.4946  TE=0.0200
  Epoch 18: QE=0.4959  TE=0.0500
  Epoch 19: QE=0.4911  TE=0.0200
  Epoch 20: QE=0.4919  TE=0.0600

IMAGE SOM
Best epoch: 17/20
QE: 0.3366
TE: 0.0200

Epoch-by-epoch:
  Epoch  1: QE=0.3363  TE=0.0500
  Epoch  2: QE=0.3379  TE=0.1100
  Epoch  3: QE=0.3366  TE=0.0600
  Epoch  4: QE=0.3373  TE=0.0700
  Epoch  5: QE=0.3363  TE=0.0900
  Epoch  6: QE=0.3392  TE=0.0900
 

In [31]:
import os
files = [f for f in os.listdir(OUT_SEARCH) if f.endswith('.png')]
print(f'Search examples saved: {len(files)}')
for f in sorted(files):
    print(f'  {f}')

Search examples saved: 20
  image_search_01.png
  image_search_02.png
  image_search_03.png
  image_search_04.png
  image_search_05.png
  image_search_06.png
  image_search_07.png
  image_search_08.png
  image_search_09.png
  image_search_10.png
  text_search_01.png
  text_search_02.png
  text_search_03.png
  text_search_04.png
  text_search_05.png
  text_search_06.png
  text_search_07.png
  text_search_08.png
  text_search_09.png
  text_search_10.png
